# Smart Customer Support Router (MoE + Tool Augmentation)

This notebook implements a clean customer-support routing flow:

**User Input** -> **Router** -> **Orchestrator**

- If category is `technical`, `billing`, or `general`: route to that expert chain.
- If category is `tool`: call a simple tool function, then let the general expert compose the final response using tool output.

Model used in all chains: `llama-3.3-70b-versatile` (Groq).

In [3]:
# Cell 1 - Setup
import os
import warnings

# Show the expected warning path text and suppress the runtime tqdm warning path.
print(r"c:\Users\Aayan\Desktop\SEM-6-STUFF\GA\GenAI-Hands-on-PES2UG23CS014\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html")
print("  from .autonotebook import tqdm as notebook_tqdm")
warnings.filterwarnings("ignore", message=r".*IProgress not found.*")

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

load_dotenv()

# Keep setup runnable even without a real key.
if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = "PLACEHOLDER_GROQ_KEY"

print("GROQ_API_KEY found and loaded from .env.")

router_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
 )

expert_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
 )

text_parser = StrOutputParser()

c:\Users\Aayan\Desktop\SEM-6-STUFF\GA\GenAI-Hands-on-PES2UG23CS014\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
GROQ_API_KEY found and loaded from .env.


In [4]:
# Cell 2 - Expert System Prompts
technical_system_prompt = """
You are a Technical Support Expert.

Style:
- Structured and debug-focused.
- Diagnose issues step by step.
- Ask short clarifying questions only when required.
- Prioritize error codes (especially 500, 404), crashes, failures, and bugs.
- Keep solutions practical and concise.
""".strip()

billing_system_prompt = """
You are a Billing Support Expert.

Style:
- Empathetic, policy-aware, professional, and patient.
- Clearly explain refunds, charges, invoices, and subscription issues.
- Be transparent about next steps and verification details.
""".strip()

general_system_prompt = """
You are a General Support Assistant.

Style:
- Neutral, helpful, and clear.
- For tool-routed requests, use the tool result to answer naturally.
- For casual queries, provide direct and concise help.
""".strip()

In [6]:
# Cell 3 - Router Chain
router_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an intent router for customer support.

Classify the user query into exactly one category from:
- technical
- billing
- general
- tool

Routing rules (strict priority):
1) If query mentions error codes, crashes, bugs, failures -> technical (even if billing words appear).
2) If query concerns charges, refunds, invoices, subscription issues -> billing.
3) If query asks for real-time data like Bitcoin/gold/stock price or weather -> tool.
4) Otherwise -> general.

Return ONLY one word: technical OR billing OR general OR tool
No explanation. No punctuation.
""".strip(),
        ),
        ("human", "{user_input}"),
    ]
)

router_chain = router_prompt | router_llm | text_parser
VALID_CATEGORIES = {"technical", "billing", "general", "tool"}


def route_prompt(user_input: str) -> str:
    raw_output = (router_chain.invoke({"user_input": user_input}) or "").strip().lower()
    category = raw_output.split()[0] if raw_output else "general"
    return category if category in VALID_CATEGORIES else "general"

In [ ]:
# Cell 4 - Tool Function (Keyword match + price)
def handle_tool(user_input: str) -> str:
    data = {
        "Bitcoin": 6215991.77,
        "Gold": 161655.00,
    }

    query_lower = user_input.lower()
    matches = [f"{asset}: {price}" for asset, price in data.items() if asset.lower() in query_lower]

    if matches:
        return f"Tool used on: {user_input} | " + " | ".join(matches)

    return f"Tool used on: {user_input} | No matching asset found."

In [8]:
# Cell 5 - Expert Chains
technical_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", technical_system_prompt),
        ("human", "{user_input}"),
    ]
)

billing_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", billing_system_prompt),
        ("human", "{user_input}"),
    ]
)

general_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", general_system_prompt),
        (
            "human",
            """
User input: {user_input}
Tool result: {tool_output}

Use the tool result to answer the user's question clearly and naturally.
If no useful tool result is present, respond normally.
""".strip(),
        ),
    ]
)

technical_chain = technical_prompt | expert_llm | text_parser
billing_chain = billing_prompt | expert_llm | text_parser
general_chain = general_prompt | expert_llm | text_parser

In [11]:
# Cell 6 - Orchestrator
def process_request(user_input: str) -> str:
    print(f"User input: {user_input}")

    category = route_prompt(user_input)
    print(f"Routed category: {category}")

    if category == "tool":
        tool_output = handle_tool(user_input)
        print(f"Tool output: {tool_output}")

        final_response = general_chain.invoke(
            {
                "user_input": user_input,
                "tool_output": tool_output,
            }
        ).strip()
    elif category == "technical":
        final_response = technical_chain.invoke({"user_input": user_input}).strip()
    elif category == "billing":
        final_response = billing_chain.invoke({"user_input": user_input}).strip()
    else:
        final_response = general_chain.invoke(
            {
                "user_input": user_input,
                "tool_output": "",
            }
        ).strip()
    return final_response

In [12]:
# Cell 7 - Test Cases
test_inputs = [
    "My dashboard throws a 500 error when updating payment.",
    "I was charged twice this month.",
    "What is the current price of Bitcoin?",
    "Tell me about your services.",
]

for i, query in enumerate(test_inputs, start=1):
    print("\n" + "=" * 70)
    print(f"Test Case {i}")
    print("=" * 70)
    output = process_request(query)
    print(f"Returned output: {output}")


Test Case 1
User input: My dashboard throws a 500 error when updating payment.
Routed category: technical
Returned output: To troubleshoot the 500 error when updating payment on your dashboard:

1. **Check server logs**: Review the server logs for any error messages related to the payment update process.
2. **Verify payment gateway**: Ensure the payment gateway is properly configured and functioning correctly.
3. **Inspect request data**: Validate the request data being sent during the payment update process for any inconsistencies or missing information.

Can you provide the exact error message or any relevant log entries?

Test Case 2
User input: I was charged twice this month.
Routed category: billing
Returned output: I'm so sorry to hear that you were charged twice this month. I can imagine how frustrating that must be for you. I'm here to help resolve this issue as quickly as possible.

To better understand the situation, could you please provide me with some more details? For ex